# Lecture 4d — Serving on Kubernetes with Minikube (3 nodes, CPU only)
**AI Operations · Module 4 · core path ≈ 20 min in class · full notebook ≈ 55 min**

**You will:** (1) check the toolchain, (2) **lint** Kubernetes manifests with Python before sending them to a cluster, (3) *simulate* the Service / readiness / self-healing behaviour locally to see the mechanism, (4) reproduce the **HPA algorithm** and tune it, (5) run the real thing on a 3-node minikube cluster (v1 + v2 canary, rolling update, rollback, blue-green, autoscaling).

> **How this notebook behaves.** Sections 1–4 run on *any* machine (pure Python + local processes). Section 5 sends real commands to minikube **only if `minikube`, `kubectl` and a Docker daemon are present**; otherwise it prints each command with a *"not executed"* note, so you can still read the full runbook. The authors could not run section 5 in their sandbox (no Docker daemon) — please report any command that needs fixing.

In [ ]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.110", "uvicorn>=0.30", "scikit-learn", "pandas", "joblib", "requests", "matplotlib", "pyyaml"], check=False)
os.makedirs("data", exist_ok=True); os.makedirs("k8s", exist_ok=True)

In [1]:
%%writefile train_models.py
"""
train_models.py  --  AIOps Module 4 (Model Serving)
Trains TWO versions of a customer-churn classifier on a synthetic, deterministic dataset.
  v1 = LogisticRegression (fast, simple baseline)      -> data/model_v1.joblib
  v2 = RandomForest       (slower, usually more accurate) -> data/model_v2.joblib
Two versions let us practise canary releases, blue-green switches and rollback later.
CPU only; runs in a few seconds.
"""
import json, argparse, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

FEATURES = ["tenure_months", "monthly_charge", "support_calls", "late_payments",
            "data_usage_gb", "contract_months", "num_products", "age"]

def make_data(n=6000, seed=7):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "tenure_months": r.integers(1, 72, n),
        "monthly_charge": r.normal(65, 20, n).clip(15, 150).round(2),
        "support_calls": r.poisson(2, n),
        "late_payments": r.poisson(0.7, n),
        "data_usage_gb": r.gamma(3, 8, n).round(2),
        "contract_months": r.choice([1, 12, 24], n, p=[.5, .3, .2]),
        "num_products": r.integers(1, 6, n),
        "age": r.integers(18, 80, n),
    })
    z = (-2.0 - 0.02*df.tenure_months + 0.01*df.monthly_charge + 0.15*df.support_calls
         + 0.2*df.late_payments - 0.01*df.contract_months - 0.1*df.num_products
         + 2.2*((df.support_calls >= 4) & (df.contract_months == 1))      # non-linear interactions:
         + 1.8*((df.tenure_months < 12) & (df.monthly_charge > 80))       # a linear model cannot
         + 1.5*((df.late_payments >= 2) & (df.num_products <= 2))         # capture these well
         - 1.6*((df.contract_months == 24) & (df.tenure_months > 36))
         + r.normal(0, .5, n))
    df["churn"] = (1/(1+np.exp(-z)) > r.uniform(size=n)).astype(int)
    return df

def main(out="data"):
    df = make_data()
    X, y = df[FEATURES], df["churn"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)
    models = {
        "v1": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "v2": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=0, n_jobs=1),
    }
    report = {}
    for ver, m in models.items():
        m.fit(Xtr, ytr)
        acc = accuracy_score(yte, m.predict(Xte))
        joblib.dump({"model": m, "feature_columns": FEATURES, "version": ver}, f"{out}/model_{ver}.joblib")
        report[ver] = round(acc, 4)
        print(f"{ver}: test accuracy = {acc:.4f}")
    df.to_csv(f"{out}/churn.csv", index=False)
    Xte.head(200).to_csv(f"{out}/sample_requests.csv", index=False)
    json.dump({"rows": Xte.head(3).to_dict(orient="records")}, open(f"{out}/sample_payload.json", "w"), indent=2)
    json.dump(report, open(f"{out}/metrics.json", "w"))

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--out", default="data"); main(ap.parse_args().out)

Overwriting train_models.py


In [2]:
%%writefile rest_app.py
"""
rest_app.py  --  AIOps Module 4, Topic 1: model serving over a REST API (FastAPI)
Run:  MODEL_PATH=data/model_v1.joblib uvicorn rest_app:app --port 8000
Endpoints
  GET  /healthz                      liveness/readiness probe
  GET  /v1/models/churn              metadata (version, features)
  POST /v1/models/churn:predict      {"instances":[{...feature dict...}, ...]}
"""
import os, time, socket, joblib
import pandas as pd
from typing import List
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

MODEL_PATH = os.getenv("MODEL_PATH", "data/model_v1.joblib")
bundle = joblib.load(MODEL_PATH)                 # loaded ONCE at startup, never per request
model, FEATURES, VERSION = bundle["model"], bundle["feature_columns"], bundle["version"]
POD = os.getenv("POD_NAME", socket.gethostname())

class Customer(BaseModel):                       # schema = contract with the client
    tenure_months: int = Field(ge=0, le=120)
    monthly_charge: float = Field(ge=0, le=500)
    support_calls: int = Field(ge=0)
    late_payments: int = Field(ge=0)
    data_usage_gb: float = Field(ge=0)
    contract_months: int = Field(ge=1, le=60)
    num_products: int = Field(ge=1, le=20)
    age: int = Field(ge=18, le=110)

class PredictRequest(BaseModel):
    instances: List[Customer] = Field(min_length=1, max_length=1000)

app = FastAPI(title="Churn model service")
STATS = {"requests": 0, "errors": 0}

@app.middleware("http")
async def timing(request: Request, call_next):   # cheap observability: latency header + counters
    t0 = time.perf_counter()
    resp = await call_next(request)
    resp.headers["X-Process-Time-ms"] = f"{(time.perf_counter()-t0)*1000:.2f}"
    STATS["requests"] += 1
    STATS["errors"] += int(resp.status_code >= 400)
    return resp

@app.get("/healthz")
def healthz():
    return {"status": "ok", "model_version": VERSION}

@app.get("/v1/models/churn")
def metadata():
    return {"name": "churn", "version": VERSION, "features": FEATURES, "served_by": POD}

@app.get("/stats")
def stats():
    return {**STATS, "served_by": POD, "model_version": VERSION}

@app.post("/v1/models/churn:predict")
def predict(req: PredictRequest):
    # plain `def` => FastAPI runs it in a thread pool, so CPU-bound sklearn does not block the event loop
    try:
        X = pd.DataFrame([c.model_dump() for c in req.instances])[FEATURES]
        proba = model.predict_proba(X)[:, 1]
    except Exception as e:                       # never leak a stack trace to the client
        raise HTTPException(status_code=500, detail=f"inference failed: {type(e).__name__}")
    return {"predictions": [int(p >= 0.5) for p in proba],
            "probabilities": [round(float(p), 4) for p in proba],
            "model_version": VERSION, "served_by": POD}

Overwriting rest_app.py


In [3]:
%%writefile load_test.py
"""
load_test.py -- generic closed-loop load generator for any model endpoint (REST / MLflow / Ray / K8s).
  python load_test.py --url http://localhost:8000/v1/models/churn:predict --format rest
  python load_test.py --url http://localhost:5001/invocations --format mlflow --concurrency 1 4 8
Reports p50/p95/p99 latency, throughput, errors and which replicas / model versions answered.
"""
import argparse, json, time, statistics, collections, concurrent.futures as cf
import pandas as pd, requests

def payload(rows, fmt, batch):
    recs = rows[:batch]
    if fmt == "rest":   return {"instances": recs}
    if fmt == "mlflow": return {"dataframe_records": recs}
    raise ValueError(fmt)

def one(sess, url, body):
    t0 = time.perf_counter()
    try:
        r = sess.post(url, json=body, timeout=30)
        ok = r.status_code == 200
        j = r.json() if ok else {}
        who = j.get("served_by", "n/a") if isinstance(j, dict) else "n/a"
        ver = j.get("model_version", "n/a") if isinstance(j, dict) else "n/a"
        if ver == "n/a" and isinstance(j, dict) and j.get("predictions") and isinstance(j["predictions"][0], dict):
            ver = j["predictions"][0].get("model_version", "n/a")      # MLflow pyfunc output
    except Exception:
        ok, who, ver = False, "n/a", "n/a"
    return (time.perf_counter() - t0) * 1000, ok, who, ver

def run(url, fmt, conc, duration, rows, batch):
    body = payload(rows, fmt, batch)
    lat, errs, who, ver = [], 0, collections.Counter(), collections.Counter()
    end = time.time() + duration
    def worker():
        s = requests.Session(); out = []
        while time.time() < end:
            out.append(one(s, url, body))
        return out
    with cf.ThreadPoolExecutor(conc) as ex:
        for res in [f.result() for f in [ex.submit(worker) for _ in range(conc)]]:
            for ms, ok, w, v in res:
                lat.append(ms); errs += (not ok); who[w] += 1; ver[v] += 1
    lat.sort(); n = len(lat); q = lambda p: lat[min(n-1, int(p*n))]
    return dict(concurrency=conc, requests=n, rps=round(n/duration, 1), p50_ms=round(q(.5), 1),
                p95_ms=round(q(.95), 1), p99_ms=round(q(.99), 1), errors=errs,
                replicas=len(who), versions=dict(ver))

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True); ap.add_argument("--format", default="rest")
    ap.add_argument("--concurrency", type=int, nargs="+", default=[1, 4, 8])
    ap.add_argument("--duration", type=float, default=5); ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--data", default="data/sample_requests.csv"); ap.add_argument("--out")
    a = ap.parse_args()
    rows = pd.read_csv(a.data).to_dict(orient="records")
    results = [run(a.url, a.format, c, a.duration, rows, a.batch) for c in a.concurrency]
    print(pd.DataFrame(results).to_string(index=False))
    if a.out: json.dump(results, open(a.out, "w"), indent=2)

Overwriting load_test.py


In [4]:
%%writefile m4util.py
"""m4util.py -- tiny helpers shared by the Module 4 notebooks (start / wait for / stop server processes)."""
import subprocess, time, os, signal, sys, requests

_procs = []

def start(cmd, log="server.log", env=None):
    """Start a server as a child process; stdout/stderr go to a log file (read it when things break)."""
    e = {**os.environ, **(env or {})}
    f = open(log, "w")
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=f, stderr=subprocess.STDOUT, env=e,
                         preexec_fn=os.setsid)
    _procs.append(p)
    return p

def wait_ready(url, timeout=90):
    """Poll a URL until it answers 200 (readiness probe, exactly what Kubernetes does)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return round(time.time() - t0, 1)
        except Exception:
            pass
        time.sleep(0.5)
    raise TimeoutError(f"{url} not ready after {timeout}s -- check the log file")

def stop(p, timeout=10):
    """Stop one server: SIGTERM the whole process group (so uvicorn/gunicorn workers die too),
    then WAIT until the group is really gone -- otherwise a restart can 'succeed' against the old process."""
    try:
        pgid = os.getpgid(p.pid)
    except Exception:
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        t0 = time.time()
        while time.time() - t0 < timeout:
            try: os.killpg(pgid, 0)            # signal 0 = "does the group still exist?"
            except ProcessLookupError: break
            time.sleep(0.2)
        else:
            os.killpg(pgid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    try: p.wait(timeout=2)
    except Exception: pass

def stop_all():
    for p in _procs: stop(p)
    _procs.clear()

Overwriting m4util.py


In [5]:
%%writefile Dockerfile.serve
# Dockerfile.serve -- AIOps Module 4 (Minikube serving). Multi-stage; the model is BAKED IN
# so every replica on every node has it with no shared storage. Build once per model version:
#   minikube image build -t churn-serve:v1 --build-opt=build-arg=MODEL_VERSION=v1 -f Dockerfile.serve .
FROM python:3.11 AS builder
WORKDIR /build
COPY requirements-serve.txt .
RUN pip install --no-cache-dir --target=/build/deps -r requirements-serve.txt

FROM python:3.11-slim
ARG MODEL_VERSION=v1
WORKDIR /app
COPY --from=builder /build/deps /usr/local/lib/python3.11/site-packages
ENV PATH="/usr/local/lib/python3.11/site-packages/bin:${PATH}" PYTHONPATH=/usr/local/lib/python3.11/site-packages
COPY rest_app.py .
COPY data/model_${MODEL_VERSION}.joblib ./model.joblib
ENV MODEL_PATH=/app/model.joblib
EXPOSE 8000
CMD ["python", "-m", "uvicorn", "rest_app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile.serve


In [ ]:
%%writefile requirements-serve.txt
fastapi>=0.110
uvicorn>=0.30
pydantic>=2.7
scikit-learn==1.9.*
pandas>=2.2
joblib>=1.4
numpy

In [ ]:
%%writefile k8s/deployment-v1.yaml
# Stable version of the churn model. Pods carry labels app=churn, version=v1.
apiVersion: apps/v1
kind: Deployment
metadata: {name: churn-v1, labels: {app: churn, version: v1}}
spec:
  replicas: 3
  strategy:
    type: RollingUpdate
    rollingUpdate: {maxSurge: 1, maxUnavailable: 0}     # never drop below desired capacity during a rollout
  selector: {matchLabels: {app: churn, version: v1}}
  template:
    metadata: {labels: {app: churn, version: v1}}
    spec:
      topologySpreadConstraints:                         # spread replicas over nodes (multi-node minikube)
      - maxSkew: 1
        topologyKey: kubernetes.io/hostname
        whenUnsatisfiable: ScheduleAnyway
        labelSelector: {matchLabels: {app: churn, version: v1}}
      containers:
      - name: server
        image: churn-serve:v1
        imagePullPolicy: Never                           # image was built/loaded INTO minikube
        ports: [{containerPort: 8000}]
        env:
        - {name: POD_NAME, valueFrom: {fieldRef: {fieldPath: metadata.name}}}
        resources:
          requests: {cpu: 100m, memory: 128Mi}
          limits: {cpu: 500m, memory: 384Mi}
        readinessProbe: {httpGet: {path: /healthz, port: 8000}, initialDelaySeconds: 3, periodSeconds: 5}
        livenessProbe: {httpGet: {path: /healthz, port: 8000}, initialDelaySeconds: 10, periodSeconds: 10}

In [ ]:
%%writefile k8s/deployment-v2.yaml
# Candidate (canary) version of the churn model. Pods carry labels app=churn, version=v1.
apiVersion: apps/v1
kind: Deployment
metadata: {name: churn-v2, labels: {app: churn, version: v2}}
spec:
  replicas: 1
  strategy:
    type: RollingUpdate
    rollingUpdate: {maxSurge: 1, maxUnavailable: 0}     # never drop below desired capacity during a rollout
  selector: {matchLabels: {app: churn, version: v2}}
  template:
    metadata: {labels: {app: churn, version: v2}}
    spec:
      topologySpreadConstraints:                         # spread replicas over nodes (multi-node minikube)
      - maxSkew: 1
        topologyKey: kubernetes.io/hostname
        whenUnsatisfiable: ScheduleAnyway
        labelSelector: {matchLabels: {app: churn, version: v2}}
      containers:
      - name: server
        image: churn-serve:v2
        imagePullPolicy: Never                           # image was built/loaded INTO minikube
        ports: [{containerPort: 8000}]
        env:
        - {name: POD_NAME, valueFrom: {fieldRef: {fieldPath: metadata.name}}}
        resources:
          requests: {cpu: 100m, memory: 128Mi}
          limits: {cpu: 500m, memory: 384Mi}
        readinessProbe: {httpGet: {path: /healthz, port: 8000}, initialDelaySeconds: 3, periodSeconds: 5}
        livenessProbe: {httpGet: {path: /healthz, port: 8000}, initialDelaySeconds: 10, periodSeconds: 10}

In [ ]:
%%writefile k8s/service.yaml
# One Service for ALL churn pods (selector has NO version label).
# => traffic splits roughly in proportion to replica counts: v1 x3 + v2 x1 ~ 75% / 25%  (canary by replicas)
apiVersion: v1
kind: Service
metadata: {name: churn, labels: {app: churn}}
spec:
  type: NodePort
  selector: {app: churn}
  ports: [{port: 80, targetPort: 8000, name: http}]

In [ ]:
%%writefile k8s/service-bluegreen.yaml
# Blue-green: the Service selects ONE version. Switch by patching the selector (instant, and reversible).
apiVersion: v1
kind: Service
metadata: {name: churn-bg, labels: {app: churn}}
spec:
  type: NodePort
  selector: {app: churn, version: v1}      # kubectl patch svc churn-bg -p '{"spec":{"selector":{"app":"churn","version":"v2"}}}'
  ports: [{port: 80, targetPort: 8000, name: http}]

In [ ]:
%%writefile k8s/hpa.yaml
# Horizontal Pod Autoscaler for the stable deployment. Needs:  minikube addons enable metrics-server
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata: {name: churn-v1}
spec:
  scaleTargetRef: {apiVersion: apps/v1, kind: Deployment, name: churn-v1}
  minReplicas: 2
  maxReplicas: 6
  metrics:
  - type: Resource
    resource: {name: cpu, target: {type: Utilization, averageUtilization: 50}}
  behavior:
    scaleDown: {stabilizationWindowSeconds: 60}

In [ ]:
!python train_models.py

## 1 · Toolchain check

In [ ]:
import shutil
def ver(cmd):
    try: return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=20).stdout.strip().splitlines()[0][:80]
    except Exception: return "-"
tools = {t: shutil.which(t) for t in ("docker", "minikube", "kubectl")}
docker_ok = tools["docker"] is not None and subprocess.run("docker info", shell=True, capture_output=True).returncode == 0
CLUSTER = bool(tools["minikube"] and tools["kubectl"] and docker_ok)
print({k: (v or "MISSING") for k, v in tools.items()}, "| docker daemon:", docker_ok)
print("=> real cluster commands will", "RUN" if CLUSTER else "be PRINTED ONLY (not executed)")

Install hints: Docker Desktop / Engine, `minikube` (https://minikube.sigs.k8s.io), `kubectl`. Budget **≈ 6 GB RAM and 6 CPUs free** for a 3-node cluster (each node `--cpus 2 --memory 2200`). If your laptop is smaller use `--nodes 2`, or run just sections 1–4.

## 2 · Lint the manifests before you apply them

Most Kubernetes outages in a first deployment are *label mismatches* and *wrong probe paths*. Both can be caught in Python in a few lines.

In [ ]:
import yaml, glob, re, pandas as pd
docs = {}
for f in sorted(glob.glob("k8s/*.yaml")):
    for d in yaml.safe_load_all(open(f)): docs[(d["kind"], d["metadata"]["name"])] = d      # key by kind+name: HPA and Deployment share a name
deployments = {n: d for (k, n), d in docs.items() if k == "Deployment"}
services = {n: d for (k, n), d in docs.items() if k == "Service"}
app_src = open("rest_app.py").read()
checks = []
for n, d in deployments.items():
    sel, tmpl = d["spec"]["selector"]["matchLabels"], d["spec"]["template"]["metadata"]["labels"]
    cont = d["spec"]["template"]["spec"]["containers"][0]
    checks += [
      (n, "selector ⊆ pod labels",              all(tmpl.get(k) == v for k, v in sel.items())),
      (n, "readiness path exists in app",        f'"{cont["readinessProbe"]["httpGet"]["path"]}"' in app_src),
      (n, "probe port == container port",        cont["readinessProbe"]["httpGet"]["port"] == cont["ports"][0]["containerPort"]),
      (n, "requests <= limits (cpu)",            int(cont["resources"]["requests"]["cpu"][:-1]) <= int(cont["resources"]["limits"]["cpu"][:-1])),
      (n, "imagePullPolicy Never (local image)", cont["imagePullPolicy"] == "Never"),
    ]
for n, s in services.items():
    sel = s["spec"]["selector"]
    matched = [dn for dn, d in deployments.items() if all(d["spec"]["template"]["metadata"]["labels"].get(k) == v for k, v in sel.items())]
    checks.append((n, f"selects pods of: {matched}", len(matched) > 0))
    checks.append((n, "targetPort == containerPort", s["spec"]["ports"][0]["targetPort"] == 8000))
pd.DataFrame(checks, columns=["object", "check", "pass"])

In [ ]:
# capacity plan: will it fit on 3 nodes of 2 CPU each?
req = sum(int(d["spec"]["replicas"]) * int(d["spec"]["template"]["spec"]["containers"][0]["resources"]["requests"]["cpu"][:-1]) for d in deployments.values())
lim = sum(int(d["spec"]["replicas"]) * int(d["spec"]["template"]["spec"]["containers"][0]["resources"]["limits"]["cpu"][:-1]) for d in deployments.values())
print(f"total CPU requests: {req}m   total limits: {lim}m   cluster: 3 nodes x 2000m = 6000m (system pods use ~15-20%)")
share = {n: d["spec"]["replicas"] for n, d in deployments.items()}; tot = sum(share.values())
print("expected traffic share through Service 'churn' (selector app=churn):", {n: f"{r/tot:.0%}" for n, r in share.items()})

The **requests** drive *scheduling* (does a pod fit on a node?) and the HPA's utilization math; **limits** drive throttling. Both numbers matter, for different reasons.

## 3 · Simulate Service, readiness and self-healing locally

We start **4 'pods'** (3 × v1, 1 × v2) as local processes and write a tiny stand-in for `kube-proxy`: a *readiness poller* maintains an **endpoint list** from pods whose `/healthz` answers, and the client sends each request to a random ready endpoint. A 'ReplicaSet' loop restarts dead pods. This is a **simulation of the mechanism** — the real components are the kubelet, the Endpoints controller, kube-proxy and the ReplicaSet controller.

In [ ]:
import threading, time, random, requests, collections, m4util
PODS = {}   # name -> dict(port, version, proc)
def spawn(name, port, version):
    p = m4util.start(f"python -m uvicorn rest_app:app --port {port}", log=f"pod_{name}.log",
                     env={"MODEL_PATH": f"data/model_{version}.joblib", "POD_NAME": name})
    PODS[name] = dict(port=port, version=version, proc=p)
layout = [("churn-v1-a", 9001, "v1"), ("churn-v1-b", 9002, "v1"), ("churn-v1-c", 9003, "v1"), ("churn-v2-a", 9004, "v2")]
for n, port, v in layout: spawn(n, port, v)
for n in PODS: m4util.wait_ready(f"http://localhost:{PODS[n]['port']}/healthz")
print("pods up:", list(PODS))

In [ ]:
ENDPOINTS, EVENTS, stop_ctl = set(), [], False
def controller():                                   # readiness probes + ReplicaSet self-healing
    t0 = time.time()
    while not stop_ctl:
        for n, pod in PODS.items():
            try: ready = requests.get(f"http://localhost:{pod['port']}/healthz", timeout=0.5).status_code == 200
            except Exception: ready = False
            if ready and n not in ENDPOINTS: ENDPOINTS.add(n); EVENTS.append((round(time.time()-t0,1), n, "added to endpoints"))
            if not ready and n in ENDPOINTS: ENDPOINTS.discard(n); EVENTS.append((round(time.time()-t0,1), n, "REMOVED from endpoints"))
            if not ready and pod["proc"].poll() is not None:     # container exited -> restart (self-healing)
                EVENTS.append((round(time.time()-t0,1), n, "restarting")); spawn(n, pod["port"], pod["version"])
        time.sleep(0.5)
ctl = threading.Thread(target=controller, daemon=True); ctl.start(); time.sleep(2)
def call(i=0):
    pool = [n for n in ENDPOINTS]
    if not pool: return None, "no endpoints"
    n = random.choice(pool); pod = PODS[n]
    try:
        r = requests.post(f"http://localhost:{pod['port']}/v1/models/churn:predict", json={"instances": [recs[i % len(recs)]]}, timeout=3)
        return r.json()["model_version"], r.json()["served_by"]
    except Exception: return None, n
recs = pd.read_csv("data/sample_requests.csv").to_dict(orient="records")
res = [call(i) for i in range(400)]
cnt = collections.Counter(v for v, _ in res); pods_seen = collections.Counter(p for _, p in res)
print("version split:", {k: f"{v/400:.0%}" for k, v in cnt.items()}); print("per pod:", dict(pods_seen))

Four ready endpoints, random choice ⇒ ≈ **25 % to v2** (1 of 4 pods): *a canary by replica ratio*. Now **kill a v1 pod** while sending traffic.

In [ ]:
import os, signal
victim = PODS["churn-v1-b"]["proc"]; os.killpg(os.getpgid(victim.pid), signal.SIGKILL)
t0 = time.time(); ok = bad = 0
while time.time() - t0 < 8:
    v, who = call(); ok += v is not None; bad += v is None; time.sleep(0.05)
print(f"requests during the failure window: {ok} ok, {bad} failed")
for e in EVENTS: print(e)

**What you saw.** Within about a probe period the dead pod left the endpoint list (clients stop being sent there), and the 'ReplicaSet' brought a replacement that rejoined only after its readiness probe passed. A few requests that were *in flight or routed in the gap* may fail — which is why production clients use retries (Lecture 4a, Task A) and why `maxUnavailable: 0` and readiness probes matter for *rollouts*.

In [ ]:
stop_ctl = True; ctl.join(); m4util.stop_all(); print("simulated pods stopped")

## 4 · The HPA algorithm — simulate and tune it

Kubernetes' Horizontal Pod Autoscaler computes, every sync period (default 15 s):

&nbsp;&nbsp;&nbsp;**desiredReplicas = ceil( currentReplicas × currentUtilization / targetUtilization )**

where utilization = CPU usage ÷ CPU **request**. It ignores changes within a 10 % tolerance, scales up immediately, and scales **down only to the highest recommendation of the last stabilization window** (60 s in our manifest). Below, one tick = one sync period.

In [ ]:
import math, matplotlib.pyplot as plt
def hpa_sim(traffic, cap_per_pod=40, target=50, min_r=2, max_r=6, window=4, start=None):
    r = start or min_r; hist, out = [], []
    for t, rps in enumerate(traffic):
        util = 100 * rps / (r * cap_per_pod)                   # % of CPU request (can exceed 100)
        want = r if abs(util / target - 1) <= 0.10 else math.ceil(r * util / target)
        want = max(min_r, min(max_r, want)); hist.append(want)
        r = want if want > r else min(r, max(hist[-window:]))   # up: immediately; down: window max
        out.append(dict(t=t, rps=rps, util=round(util), replicas=r))
    return pd.DataFrame(out)
traffic = [20]*4 + list(range(20, 301, 20)) + [300]*6 + [60]*14       # ramp up, hold, drop
sim = hpa_sim(traffic)
fig, ax = plt.subplots(figsize=(8, 3)); ax.plot(sim.t, sim.rps, label="traffic (rps)", c="gray"); ax2 = ax.twinx()
ax2.step(sim.t, sim.replicas, where="post", c="tab:blue", label="replicas"); ax.set_xlabel("HPA tick (15 s)"); ax.set_ylabel("rps"); ax2.set_ylabel("replicas")
plt.title("HPA reaction to a traffic ramp"); plt.tight_layout(); plt.show()
sim.iloc[[0, 6, 10, 14, 18, 24, 28, 34]]

Notice (1) the replica count **lags** the ramp, (2) it caps at `maxReplicas` while utilization stays above target — capacity planning, not magic — and (3) it holds the high count for the stabilization window after traffic drops.

## 5 · The real cluster (executes only if minikube + kubectl + Docker are available)

In [ ]:
def sh(cmd, timeout=900):
    print("$", cmd)
    if not CLUSTER: print("   [not executed: minikube/kubectl/Docker daemon not available here]\n"); return None
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    print((r.stdout + r.stderr).strip()[-1500:], "\n"); return r

### 5.1 Create a 3-node cluster and build images *into* it

In [ ]:
sh("minikube start -p aiops --nodes 3 --cpus 2 --memory 2200 --driver=docker")
sh("minikube -p aiops addons enable metrics-server")
sh("kubectl get nodes -o wide")
sh("minikube -p aiops image build -t churn-serve:v1 --build-opt=build-arg=MODEL_VERSION=v1 -f Dockerfile.serve .")
sh("minikube -p aiops image build -t churn-serve:v2 --build-opt=build-arg=MODEL_VERSION=v2 -f Dockerfile.serve .")
sh("minikube -p aiops image ls | grep churn-serve")

`minikube image build` builds **inside** the cluster's container runtime (no registry needed) — hence `imagePullPolicy: Never` in the manifests. With 3 nodes `minikube image build` loads the image on all nodes; if a pod ever shows `ErrImageNeverPull`, run `minikube -p aiops image load churn-serve:v1` for that image.

### 5.2 Deploy v1 ×3 and v2 ×1 behind one Service (canary by replica ratio)

In [ ]:
sh("kubectl apply -f k8s/deployment-v1.yaml -f k8s/deployment-v2.yaml -f k8s/service.yaml")
sh("kubectl rollout status deploy/churn-v1 --timeout=180s && kubectl rollout status deploy/churn-v2 --timeout=180s")
sh("kubectl get pods -l app=churn -o wide")      # NODE column: are replicas spread over 3 nodes?

In [ ]:
# reach the NodePort Service from the host (docker driver) -- keep this tunnel open in a terminal instead of a notebook cell:
#     minikube -p aiops service churn --url
r = sh("minikube -p aiops service churn --url")
URL = (r.stdout.strip().splitlines()[0] if r is not None and r.returncode == 0 else "http://127.0.0.1:<PORT-FROM-ABOVE>") + "/v1/models/churn:predict"
print(URL)
if CLUSTER:
    import load_test
    rows_all = pd.read_csv("data/sample_requests.csv").to_dict(orient="records")
    print(pd.DataFrame([load_test.run(URL, "rest", cc, 5, rows_all, 1) for cc in (1, 4, 8)]))

The `versions` column of the load-test table should show ≈ 75 % v1 / 25 % v2, and `replicas` the number of *distinct pods* that answered.

### 5.3 Scale, self-heal and spread

In [ ]:
sh("kubectl scale deploy/churn-v1 --replicas=6")
sh("kubectl get pods -l app=churn,version=v1 -o wide")
sh("kubectl delete pod -l app=churn,version=v2 --wait=false && sleep 3 && kubectl get pods -l version=v2")   # self-healing

### 5.4 Rolling update and rollback

In [ ]:
sh("kubectl set image deploy/churn-v1 server=churn-serve:v2")             # update v1's pods to the v2 image
sh("kubectl rollout status deploy/churn-v1 --timeout=180s")
sh("kubectl rollout history deploy/churn-v1")
sh("kubectl rollout undo deploy/churn-v1")                                # ROLLBACK to the previous ReplicaSet
sh("kubectl rollout status deploy/churn-v1 --timeout=180s")

With `maxSurge: 1, maxUnavailable: 0` Kubernetes starts one new pod, waits for **readiness**, only then removes an old one — capacity never dips. A *broken* image never becomes Ready, so the rollout stalls and the old pods keep serving: that is your safety net (try `kubectl set image deploy/churn-v1 server=churn-serve:does-not-exist`, watch `rollout status`, then `rollout undo`).

### 5.5 Blue-green switch

In [ ]:
sh("kubectl apply -f k8s/service-bluegreen.yaml")
sh('''kubectl patch svc churn-bg -p '{"spec":{"selector":{"app":"churn","version":"v2"}}}' ''')      # switch to v2 (green)
sh('''kubectl patch svc churn-bg -p '{"spec":{"selector":{"app":"churn","version":"v1"}}}' ''')      # instant rollback to v1 (blue)
sh("kubectl get endpoints churn-bg")

### 5.6 Autoscaling under load

In [ ]:
sh("kubectl apply -f k8s/hpa.yaml")
sh("kubectl get hpa churn-v1")
# in a terminal:  python load_test.py --url <URL> --concurrency 16 --duration 120     then:   kubectl get hpa churn-v1 -w

In [ ]:
sh("kubectl top pods -l app=churn")
sh("kubectl describe hpa churn-v1 | tail -15")
# clean up when finished:
# minikube delete -p aiops

## 6 · YOUR TURN (runs anywhere)

**Task A — canary planner.** Write `plan_canary(total_pods, target_pct)` returning `(v1_pods, v2_pods)` so the Service's replica-ratio split is as close as possible to `target_pct` % for v2, keeping **at least 1 pod of each** version. Report the achieved percentage. What is the **smallest canary** achievable with 4 pods in total — and what does that tell you about canary-by-replicas for small services?

**Task B — size the HPA.** Using `hpa_sim`, find the smallest `max_r` (2–10) such that, for the traffic profile above with `cap_per_pod=40` and `target=50`, utilization at the end of the 300 rps plateau is at most **100 %**. Then say what changes if `window=1` (shorter scale-down stabilization).

In [ ]:
# Task A — your code here

**Solution A**

In [ ]:
def plan_canary(total_pods, target_pct):
    best = None
    for v2 in range(1, total_pods):
        achieved = 100 * v2 / total_pods
        if best is None or abs(achieved - target_pct) < abs(best[2] - target_pct): best = (total_pods - v2, v2, achieved)
    return best[0], best[1], round(best[2], 1)
for n, pct in [(4, 10), (10, 10), (20, 5), (50, 5)]:
    print(f"{n:>2} pods, want {pct:>2}% -> (v1, v2, achieved%) = {plan_canary(n, pct)}")
print("smallest canary with 4 pods:", plan_canary(4, 0)[2], "%  -> to get a 1-5% canary you need 20-100 pods, or a mesh/ingress that splits by weight (e.g. Ray Serve router, Istio, NGINX canary annotations).")

In [ ]:
# Task B — your code here

**Solution B**

In [ ]:
for m in range(2, 11):
    s = hpa_sim(traffic, max_r=m)
    plateau = s[s.rps == 300].util.iloc[-1]          # utilization at the end of the 300-rps plateau
    print(f"max_r={m:>2}  utilization at end of the 300 rps plateau = {plateau:>3}%", "<= 100%  OK" if plateau <= 100 else "")
    if plateau <= 100: break
print("replicas right after the traffic drop (ticks 26-33)  window=1:", hpa_sim(traffic, max_r=m, window=1).replicas.tolist()[26:34])
print("                                                     window=4:", hpa_sim(traffic, max_r=m, window=4).replicas.tolist()[26:34])

A short window releases pods quickly (cheaper) but risks **flapping** if traffic oscillates; a long window is safer and costs more. Your SLO and budget decide.

### Recap
* Deployment = desired replicas, rolling updates, rollback history; **Service** = stable name + load-balancing across *ready* pods.
* **Readiness** gates traffic; **liveness** restarts; both must target an endpoint your app really serves.
* Canary by replica ratio is coarse (minimum share = 1/N); blue-green by selector patch is instant and reversible.
* **HPA** = ceil(current × util/target) with tolerance, lag and stabilization — plan `maxReplicas` for peak.

**Module 4 wrap-up:** you served one model four ways — hand-written REST, MLflow-generated REST, Ray Serve, Kubernetes — and saw the same operational ideas (health, versions, canary, rollback, autoscaling) recur at each layer.